In [1]:
import glob
import os
import re
import sys
import numpy as np
import optuna
import pandas as pd

In [2]:
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

from drGAT.metrics import get_parsed_df

In [3]:
def extract_method_data(filename):
    match = re.match(r"([A-Za-zA-Z0-9]+)_([a-z0-9]+)\.sqlite3", filename)
    return match.groups() if match else ("unknown", "unknown")

In [62]:
all_dfs = []

for file in glob.glob("*.sqlite3"):
    method, data = extract_method_data(os.path.basename(file))
    try:
        study = optuna.load_study(study_name=method, storage=f"sqlite:///{file}")
        df_all = study.trials_dataframe()

        # ✅ COMPLETE 状態の数を表示
        n_complete = (df_all["state"] == "COMPLETE").sum()
        print(f"✅ {file}: {n_complete} trials completed")

        df_valid = df_all.dropna(
            subset=["values_0", "values_1", "values_2", "values_3"]
        )

        if df_valid.shape[0] == 0:
            continue

        # user_attrs から評価指標を取り出す
        df_metrics = df_valid.iloc[:, 20:-2]
        df_metrics.columns = [c.replace("user_attrs_", "") for c in df_metrics.columns]
        parsed_df = get_parsed_df(df_metrics)
        parsed_df["n_complete"] = n_complete

        # ハイパーパラメータ列だけ抽出
        df_params = df_valid[
            [c for c in df_valid.columns if "params" in c]
        ].reset_index(drop=True)
        parsed_df = pd.concat([parsed_df.reset_index(drop=True), df_params], axis=1)

        # method / data 列を追加
        parsed_df["method"] = method
        parsed_df["data"] = data

        all_dfs.append(parsed_df)

    except Exception as e:
        print(f"❌ Failed to parse {file}: {e}")

if all_dfs:
    summary_df = pd.concat(all_dfs, ignore_index=True)
    summary_df["AUPR_num"] = summary_df["AUPR"].str.extract(r"([\d\.]+)").astype(float)

    # 最良インデックス取得
    best_idxs = summary_df.groupby(["data", "method"])["AUPR_num"].idxmax()
    best_df = summary_df.loc[best_idxs].drop(columns=["AUPR_num"])

    # ✅ インデックス設定: method, data の順で MultiIndex
    best_df.set_index(
        [
            "data",
            "method",
        ],
        inplace=True,
    )

    # ✅ data をキーに並べ替え（index の level 1 をソート）
    best_df.sort_index(level="data", inplace=True)

    # 任意の順に並べたいカラム
    desired_order = ["n_complete", "ACC", "Precision", "Recall", "F1", "AUROC", "AUPR"]

    # その順に並び替える（残りのカラムはそのまま後ろへ）
    other_cols = [col for col in best_df.columns if col not in desired_order]
    best_df = best_df[desired_order + other_cols]

    # インデックスの data レベルを並び替え
    desired_data_order = [
        "nci",
        "gdsc1",
        "gdsc2",
        "ctrp",
    ]  # 小文字なら小文字にそろえる！

    # sort_index ではなく reindex で順序を明示
    best_df = best_df.reindex(
        best_df.index.reindex(
            sorted(best_df.index, key=lambda x: desired_data_order.index(x[0]))
        )[0]
    )

    display(best_df)
else:
    print("No valid studies found.")

✅ GAT_ctrp.sqlite3: 704 trials completed
✅ GAT_gdsc1.sqlite3: 654 trials completed
✅ GAT_gdsc2.sqlite3: 735 trials completed
✅ GAT_nci.sqlite3: 2397 trials completed
✅ GATv2_ctrp.sqlite3: 678 trials completed
✅ GATv2_gdsc1.sqlite3: 674 trials completed
✅ GATv2_gdsc2.sqlite3: 632 trials completed
✅ GATv2_nci.sqlite3: 2625 trials completed
✅ Transformer_ctrp.sqlite3: 521 trials completed
✅ Transformer_gdsc1.sqlite3: 470 trials completed
❌ Failed to parse Broken_Transformer_gdsc2.sqlite3: 'Record does not exist.'
✅ Transformer_nci.sqlite3: 2478 trials completed
❌ Failed to parse Transformer_gdsc2_recovered.sqlite3: 'Record does not exist.'
✅ Transformer_gdsc2.sqlite3: 0 trials completed


n_complete              ACC        Precision  \
data  method                                                      
nci   GAT                2397  0.876 (± 0.007)  0.914 (± 0.008)   
      GATv2              2625  0.875 (± 0.006)  0.915 (± 0.014)   
      Transformer        2478  0.868 (± 0.009)  0.915 (± 0.011)   
gdsc1 GAT                 654   0.79 (± 0.007)   0.787 (± 0.01)   
      GATv2               674  0.784 (± 0.009)  0.773 (± 0.016)   
      Transformer         470  0.803 (± 0.004)  0.794 (± 0.012)   
gdsc2 GAT                 735  0.805 (± 0.008)   0.81 (± 0.011)   
      GATv2               632  0.815 (± 0.012)   0.82 (± 0.008)   
ctrp  GAT                 704  0.848 (± 0.004)   0.853 (± 0.01)   
      GATv2               678  0.848 (± 0.004)  0.855 (± 0.005)   
      Transformer         521  0.861 (± 0.006)  0.867 (± 0.004)   

                            Recall               F1            AUROC  \
data  method                                                           
nci   GAT          0.826 (± 0.012)  0.868 (± 0.007)  0.948 (± 0.003)   
      GATv2        0.824 (± 0.017)  0.867 (± 0.006)  0.948 (± 0.003)   
      Transformer  0.807 (± 0.023)  0.857 (± 0.012)  0.945 (± 0.003)   
gdsc1 GAT           0.78 (± 0.014)  0.784 (± 0.006)  0.867 (± 0.007)   
      GATv2        0.791 (± 0.011)  0.781 (± 0.008)  0.866 (± 0.006)   
      Transformer  0.805 (± 0.009)  0.799 (± 0.004)  0.884 (± 0.002)   
gdsc2 GAT          0.817 (± 0.018)   0.813 (± 0.01)  0.885 (± 0.006)   
      GATv2        0.825 (± 0.027)  0.822 (± 0.015)   0.892 (± 0.01)   
ctrp  GAT          0.854 (± 0.009)  0.854 (± 0.004)  0.924 (± 0.002)   
      GATv2        0.852 (± 0.007)  0.853 (± 0.003)  0.927 (± 0.002)   
      Transformer  0.866 (± 0.014)  0.867 (± 0.006)  0.935 (± 0.003)   

                              AUPR     Balanced_ACC  params_T_max  \
data  method                                                        
nci   GAT          0.946 (± 0.004)  0.875 (± 0.007)           NaN   
      GATv2        0.947 (± 0.003)  0.875 (± 0.006)           NaN   
      Transformer  0.943 (± 0.004)   0.867 (± 0.01)           NaN   
gdsc1 GAT          0.869 (± 0.007)   0.79 (± 0.007)           NaN   
      GATv2        0.867 (± 0.006)  0.785 (± 0.009)           NaN   
      Transformer  0.885 (± 0.004)  0.803 (± 0.004)           NaN   
gdsc2 GAT          0.889 (± 0.013)  0.805 (± 0.008)           NaN   
      GATv2        0.895 (± 0.011)  0.815 (± 0.012)           NaN   
ctrp  GAT          0.933 (± 0.003)  0.847 (± 0.004)           NaN   
      GATv2        0.936 (± 0.003)  0.847 (± 0.004)           NaN   
      Transformer  0.942 (± 0.002)  0.861 (± 0.006)           NaN   

                  params_activation  ...  params_hidden1  params_hidden2  \
data  method                         ...                                   
nci   GAT                      relu  ...             259              96   
      GATv2                    relu  ...             395             236   
      Transformer              relu  ...             289             215   
gdsc1 GAT                      gelu  ...             325             236   
      GATv2                    gelu  ...             400              85   
      Transformer              gelu  ...             283             134   
gdsc2 GAT                      gelu  ...             496              72   
      GATv2                    relu  ...             377             103   
ctrp  GAT                      gelu  ...             398              83   
      GATv2                    gelu  ...             436              93   
      Transformer              gelu  ...             368             153   

                   params_hidden3  params_is_zero_pad  params_lr  \
data  method                                                       
nci   GAT                      91                True   0.000265   
      GATv2                    80                True   0.000376   
      Transformer              66                Tr

In [99]:
tmp.columns

Index(['data', 'method', 'params_T_max', 'params_activation',
       'params_attention_dropout', 'params_dropout1', 'params_dropout2',
       'params_dropout3', 'params_epochs', 'params_final_mlp_layers',
       'params_heads', 'params_hidden1', 'params_hidden2', 'params_hidden3',
       'params_is_zero_pad', 'params_lr', 'params_n_layers',
       'params_norm_type', 'params_optimizer', 'params_scheduler',
       'params_weight_decay'],
      dtype='object')

In [89]:
tmp = best_df[[i for i in best_df.columns if 'params' in i]].copy()
tmp = tmp.reset_index()

,0,1,2,3,4,5,6,7,8,9,10
data,nci,nci,nci,gdsc1,gdsc1,gdsc1,gdsc2,gdsc2,ctrp,ctrp,ctrp
method,GAT,GATv2,Transformer,GAT,GATv2,Transformer,GAT,GATv2,GAT,GATv2,Transformer
params_T_max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
params_activation,relu,relu,relu,gelu,gelu,gelu,gelu,relu,gelu,gelu,gelu
params_attention_dropout,0.2,0.2,0.1,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0
params_dropout1,0.1,0.2,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1
params_dropout2,0.1,0.1,0.2,0.1,0.4,0.3,0.2,0.1,0.2,0.2,0.1
params_dropout3,0.5,0.5,0.5,0.4,0.5,0.5,0.5,0.5,0.5,0.5,0.5
params_epochs,1000,1000,1000,800,1000,1000,1000,900,800,1000,1000
params_final_mlp_layers,2,2,2,2,2,2,2,2,3,2,2


In [102]:
def update_epochs_to_1500_and_add_layer(config_dict):
    """
    Update 'epochs' to 1500, remove 'params_' prefix from keys,
    exclude 'is_zero_pad', and add 'gnn_layer' based on the method name.
    """
    updated = {}
    for data, methods in config_dict.items():
        updated[data] = {}
        for method, params in methods.items():
            new_params = {}
            for k, v in params.items():
                # Remove 'params_' prefix
                key = k.replace("params_", "") if k.startswith("params_") else k
                if key == "is_zero_pad":
                    continue  # Exclude
                new_params[key] = v
            new_params["epochs"] = 1500  # Force override
            new_params["gnn_layer"] = method  # Add method name as param
            if "scheduler" not in new_params:
                new_params["scheduler"] = "None"  # 明示的に追加（文字列）
            updated[data][method] = new_params
    return updated



# YAML ファイルに保存も可能に
def save_config_to_yaml(config_dict, path="../configs/test1_params.yaml"):
    import yaml
    with open(path, 'w') as f:
        yaml.dump(config_dict, f, sort_keys=False)
    return path

In [103]:
config = {
    data: {
        row['method']: row.drop(labels=['data', 'method']).dropna().to_dict()
        for _, row in group.iterrows()
    }
    for data, group in tmp.groupby('data')
}
config = update_epochs_to_1500_and_add_layer(config)
yaml_path = save_config_to_yaml(config)

In [97]:
config

{'ctrp': {'GAT': {'activation': 'gelu',
   'attention_dropout': 0.0,
   'dropout1': 0.1,
   'dropout2': 0.2,
   'dropout3': 0.5,
   'epochs': 1500,
   'final_mlp_layers': 3,
   'heads': 6,
   'hidden1': 398,
   'hidden2': 83,
   'hidden3': 54,
   'lr': 0.0008575568077138319,
   'n_layers': 2,
   'norm_type': 'GraphNorm',
   'optimizer': 'Adam',
   'weight_decay': 2.793149007230527e-06,
   'gnn_layer': 'GAT'},
  'GATv2': {'activation': 'gelu',
   'attention_dropout': 0.0,
   'dropout1': 0.1,
   'dropout2': 0.2,
   'dropout3': 0.5,
   'epochs': 1500,
   'final_mlp_layers': 2,
   'heads': 4,
   'hidden1': 436,
   'hidden2': 93,
   'hidden3': 47,
   'lr': 0.001936411075355028,
   'n_layers': 3,
   'norm_type': 'GraphNorm',
   'optimizer': 'AdamW',
   'weight_decay': 2.2267039384631575e-06,
   'gnn_layer': 'GATv2'},
  'Transformer': {'activation': 'gelu',
   'attention_dropout': 0.0,
   'dropout1': 0.1,
   'dropout2': 0.1,
   'dropout3': 0.5,
   'epochs': 1500,
   'final_mlp_layers': 2,
   

In [39]:
data='gdsc2'
method='Transformer'
study = optuna.load_study(study_name=method, storage="sqlite:///Transformer_gdsc2_recovered.sqlite3")
df_all = study.trials_dataframe()
df_valid = df_all.dropna(
    subset=["values_0", "values_1", "values_2", "values_3"]
)

# user_attrs から評価指標を取り出す
df_metrics = df_valid.iloc[:, 8:-2]
df_metrics.columns = [c.replace("user_attrs_", "") for c in df_metrics.columns]
parsed_df = get_parsed_df(df_metrics)
parsed_df["n_complete"] = n_complete

# ハイパーパラメータ列だけ抽出
df_params = df_valid[
    [c for c in df_valid.columns if "params" in c]
].reset_index(drop=True)
parsed_df = pd.concat([parsed_df.reset_index(drop=True), df_params], axis=1)

# method / data 列を追加
parsed_df["method"] = method
parsed_df["data"] = data

In [40]:
parsed_df["AUPR_num"] = parsed_df["AUPR"].str.extract(r"([\d\.]+)").astype(float)

# 最良インデックス取得
best_idxs = parsed_df.groupby(["data", "method"])["AUPR_num"].idxmax()
best_df = parsed_df.loc[best_idxs].drop(columns=["AUPR_num"])

desired_order = ["n_complete", "ACC", "Precision", "Recall", "F1", "AUROC", "AUPR"]

# その順に並び替える（残りのカラムはそのまま後ろへ）
other_cols = [col for col in best_df.columns if col not in desired_order]
best_df = best_df[desired_order + other_cols]
best_df

,n_complete,ACC,Precision,Recall,F1,AUROC,AUPR,Balanced_ACC,method,data
338,2478,0.819 (± 0.012),0.815 (± 0.024),0.843 (± 0.02),0.828 (± 0.011),0.895 (± 0.011),0.899 (± 0.016),0.818 (± 0.012),Transformer,gdsc2


In [8]:
all_dfs = []

for file in glob.glob("*.sqlite3"):
    method, data = extract_method_data(os.path.basename(file))
    try:
        study = optuna.load_study(study_name=method, storage=f"sqlite:///{file}")
        df_all = study.trials_dataframe()

        # ✅ COMPLETE 状態の数を表示
        n_complete = (df_all["state"] == "COMPLETE").sum()
        print(f"✅ {file}: {n_complete} trials completed")

        df_valid = df_all.dropna(
            subset=["values_0", "values_1", "values_2", "values_3"]
        )
        df_valid = df_valid[df_valid['params_is_zero_pad'] == False]

        if df_valid.shape[0] == 0:
            continue

        # user_attrs から評価指標を取り出す
        df_metrics = df_valid.iloc[:, 20:-2]
        df_metrics.columns = [c.replace("user_attrs_", "") for c in df_metrics.columns]
        parsed_df = get_parsed_df(df_metrics)
        parsed_df["n_complete"] = n_complete

        # ハイパーパラメータ列だけ抽出
        df_params = df_valid[
            [c for c in df_valid.columns if "params" in c]
        ].reset_index(drop=True)
        parsed_df = pd.concat([parsed_df.reset_index(drop=True), df_params], axis=1)

        # method / data 列を追加
        parsed_df["method"] = method
        parsed_df["data"] = data

        all_dfs.append(parsed_df)

    except Exception as e:
        print(f"❌ Failed to parse {file}: {e}")

if all_dfs:
    summary_df = pd.concat(all_dfs, ignore_index=True)
    summary_df["AUPR_num"] = summary_df["AUPR"].str.extract(r"([\d\.]+)").astype(float)

    # 最良インデックス取得
    best_idxs = summary_df.groupby(["data", "method"])["AUPR_num"].idxmax()
    best_df = summary_df.loc[best_idxs].drop(columns=["AUPR_num"])

    # ✅ インデックス設定: method, data の順で MultiIndex
    best_df.set_index(
        [
            "data",
            "method",
        ],
        inplace=True,
    )

    # ✅ data をキーに並べ替え（index の level 1 をソート）
    best_df.sort_index(level="data", inplace=True)

    # 任意の順に並べたいカラム
    desired_order = ["n_complete", "ACC", "Precision", "Recall", "F1", "AUROC", "AUPR"]

    # その順に並び替える（残りのカラムはそのまま後ろへ）
    other_cols = [col for col in best_df.columns if col not in desired_order]
    best_df = best_df[desired_order + other_cols]

    # インデックスの data レベルを並び替え
    desired_data_order = [
        "nci",
        "gdsc1",
        "gdsc2",
        "ctrp",
    ]  # 小文字なら小文字にそろえる！

    # sort_index ではなく reindex で順序を明示
    best_df = best_df.reindex(
        best_df.index.reindex(
            sorted(best_df.index, key=lambda x: desired_data_order.index(x[0]))
        )[0]
    )

    display(best_df)
else:
    print("No valid studies found.")

✅ GAT_ctrp.sqlite3: 704 trials completed
✅ GAT_gdsc1.sqlite3: 654 trials completed
✅ GAT_gdsc2.sqlite3: 735 trials completed
✅ GAT_nci.sqlite3: 2397 trials completed
✅ GATv2_ctrp.sqlite3: 678 trials completed
✅ GATv2_gdsc1.sqlite3: 674 trials completed
✅ GATv2_gdsc2.sqlite3: 632 trials completed
✅ GATv2_nci.sqlite3: 2625 trials completed
✅ Transformer_ctrp.sqlite3: 521 trials completed
✅ Transformer_gdsc1.sqlite3: 470 trials completed
❌ Failed to parse Broken_Transformer_gdsc2.sqlite3: 'Record does not exist.'
✅ Transformer_nci.sqlite3: 2478 trials completed
❌ Failed to parse Transformer_gdsc2_recovered.sqlite3: 'Record does not exist.'
✅ Transformer_gdsc2.sqlite3: 87 trials completed


n_complete              ACC        Precision  \
data  method                                                      
nci   GAT                2397  0.781 (± 0.044)  0.821 (± 0.052)   
      GATv2              2625  0.679 (± 0.155)  0.499 (± 0.459)   
      Transformer        2478  0.759 (± 0.009)  0.811 (± 0.038)   
gdsc1 GAT                 654    0.77 (± 0.01)  0.772 (± 0.018)   
      GATv2               674  0.778 (± 0.011)  0.779 (± 0.013)   
      Transformer         470   0.786 (± 0.01)  0.783 (± 0.012)   
gdsc2 GAT                 735  0.805 (± 0.008)   0.81 (± 0.011)   
      GATv2               632  0.815 (± 0.012)   0.82 (± 0.008)   
      Transformer          87    0.8 (± 0.011)   0.806 (± 0.02)   
ctrp  GAT                 704  0.792 (± 0.044)  0.822 (± 0.049)   
      GATv2               678  0.768 (± 0.031)  0.759 (± 0.042)   
      Transformer         521  0.817 (± 0.015)  0.833 (± 0.018)   

                            Recall               F1            AUROC  \
data  method                                                           
nci   GAT          0.712 (± 0.085)   0.76 (± 0.054)   0.859 (± 0.04)   
      GATv2        0.446 (± 0.424)  0.463 (± 0.426)  0.715 (± 0.193)   
      Transformer    0.67 (± 0.06)  0.731 (± 0.023)   0.82 (± 0.019)   
gdsc1 GAT           0.75 (± 0.023)  0.761 (± 0.012)   0.849 (± 0.01)   
      GATv2         0.76 (± 0.013)  0.769 (± 0.011)  0.858 (± 0.009)   
      Transformer  0.775 (± 0.018)  0.779 (± 0.011)   0.869 (± 0.01)   
gdsc2 GAT          0.817 (± 0.018)   0.813 (± 0.01)  0.885 (± 0.006)   
      GATv2        0.825 (± 0.027)  0.822 (± 0.015)   0.892 (± 0.01)   
      Transformer   0.81 (± 0.006)   0.808 (± 0.01)  0.884 (± 0.009)   
ctrp  GAT           0.769 (± 0.05)  0.794 (± 0.044)  0.874 (± 0.044)   
      GATv2        0.817 (± 0.023)  0.786 (± 0.023)  0.856 (± 0.032)   
      Transformer  0.812 (± 0.011)  0.822 (± 0.013)  0.897 (± 0.013)   

                              AUPR     Balanced_ACC  params_T_max  \
data  method                                                        
nci   GAT          0.854 (± 0.035)   0.78 (± 0.044)           NaN   
      GATv2          0.708 (± 0.2)  0.674 (± 0.162)           NaN   
      Transformer  0.825 (± 0.013)   0.757 (± 0.01)           NaN   
gdsc1 GAT          0.854 (± 0.009)    0.77 (± 0.01)           NaN   
      GATv2         0.861 (± 0.01)  0.778 (± 0.011)           NaN   
      Transformer  0.871 (± 0.009)   0.786 (± 0.01)           NaN   
gdsc2 GAT          0.889 (± 0.013)  0.805 (± 0.008)           NaN   
      GATv2        0.895 (± 0.011)  0.815 (± 0.012)           NaN   
      Transformer   0.89 (± 0.011)    0.8 (± 0.012)           NaN   
ctrp  GAT          0.884 (± 0.044)  0.794 (± 0.044)           NaN   
      GATv2        0.868 (± 0.032)  0.766 (± 0.032)           NaN   
      Transformer  0.907 (± 0.012)  0.817 (± 0.016)           NaN   

                  params_activation  ...  params_hidden1  params_hidden2  \
data  method                         ...                                   
nci   GAT                      relu  ...             461             108   
      GATv2                    gelu  ...             299              91   
      Transformer              gelu  ...             503              80   
gdsc1 GAT                      gelu  ...             325              67   
      GATv2                    gelu  ...             362              85   
      Transformer              gelu  ...             404             135   
gdsc2 GAT                      gelu  ...             496              72   
      GATv2                    relu  ...             377             103   
      Transformer              gelu  ...             310             172   
ctrp  GAT                      gelu  ...             398              83   
      GATv2                    relu  ...             309              77   
      Transformer              gelu  ...             368             220   

                   params_hidden3  para

In [10]:
best_df.iloc[-1]

n_complete                              521
ACC                         0.817 (± 0.015)
Precision                   0.833 (± 0.018)
Recall                      0.812 (± 0.011)
F1                          0.822 (± 0.013)
AUROC                       0.897 (± 0.013)
AUPR                        0.907 (± 0.012)
Balanced_ACC                0.817 (± 0.016)
params_T_max                            NaN
params_activation                      gelu
params_attention_dropout                0.0
params_dropout1                         0.1
params_dropout2                         0.4
params_dropout3                         0.1
params_epochs                           800
params_final_mlp_layers                   2
params_heads                              2
params_hidden1                          368
params_hidden2                          220
params_hidden3                           38
params_is_zero_pad                    False
params_lr                          0.000385
params_n_layers                 